In [46]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [47]:
# Load Dataset
df = pd.read_csv("Stroke_Prediction.csv")

In [48]:
print(df.head())

      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  


In [49]:
# Drop unnecessary column
df = df.drop(columns=["id"])

In [50]:
# Fill missing values
df["bmi"] = df["bmi"].fillna(df["bmi"].mean())

In [51]:
# Features and Target
x = df.drop(columns=["stroke"])
y = df["stroke"]


In [52]:
# Train Test Split
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [53]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            [
                "gender",
                "ever_married",
                "work_type",
                "Residence_type",
                "smoking_status"
            ]
        )
    ],
    remainder="passthrough"
)


In [54]:
# Pipeline
main_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)



In [55]:
# Train Model
main_pipeline.fit(xtrain, ytrain)

c:\Users\sumai\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'ever_married',
                                                   'work_type',
                                                   'Residence_type',
                                                   'smoking_status'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [56]:
# Prediction
y_pred = main_pipeline.predict(xtest)

In [57]:
# Evaluation
print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))
print(confusion_matrix(ytest, y_pred))

Accuracy: 0.9393346379647749
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       960
           1       0.00      0.00      0.00        62

    accuracy                           0.94      1022
   macro avg       0.47      0.50      0.48      1022
weighted avg       0.88      0.94      0.91      1022

[[960   0]
 [ 62   0]]


c:\Users\sumai\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\sumai\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\sumai\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# ***FINAL CONCLUSION***:

The Random Forest Classifier learns the relationship between the patient features and stroke occurrence to predict whether a person is likely to have a stroke.

The dataset was preprocessed by filling missing BMI values, converting categorical features using One-Hot Encoding, and using class balancing to handle the imbalanced dataset.

The model was evaluated using Accuracy, Classification Report, and Confusion Matrix. Accuracy shows the overall performance, while Precision, Recall, and F1-Score measure how well the model identifies stroke and non-stroke cases.

Although the model may achieve high accuracy, the dataset is imbalanced because it contains many more non-stroke cases than stroke cases. Therefore, accuracy alone is not sufficient to judge the model's performance. The recall for stroke patients is especially important because missing a stroke case can have serious consequences.

Therefore, the model should be evaluated based on Recall, Precision, and F1-Score in addition to Accuracy. Further improvements, such as hyperparameter tuning, feature engineering, or advanced sampling techniques, can improve the model's ability to detect stroke cases more accurately.